In [1]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

In [10]:
df = pd.read_csv('dataset.csv')
df.dropna(subset=['Description', 'Class'], inplace=True)

C:\Users\Prasanna\AppData\Local\Temp\ipykernel_36788\2149519594.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('dataset.csv')


In [11]:
values_to_drop = ['A', '200', 'B']
df = df[~df['Class'].isin(values_to_drop)]

In [12]:
df['Class'] = df['Class'].astype(str)
X = df['Description']
y = df['Class']


In [70]:
X

0        Medical imaging apparatus for use in the field...
1        Medical diagnostic apparatus for testing {spec...
2                                      Balancing bird toys
3                                  Image intensifier tubes
4                                      Sunglasses for pets
                               ...                        
68055                              Fruitwood lump charcoal
68056    Hydrocarbon gas liquids (HGL) for use as fuel ...
68057    Liquefied hydrocarbon gas (LHG) for use as fue...
68058                            Natural hardwood charcoal
68059    Conducting entertainment exhibition events in ...
Name: Description, Length: 68056, dtype: object

In [13]:
# Text preprocessing
voc_size = 5000
ps = PorterStemmer()
nltk.download('stopwords')
corpus=[]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Prasanna\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
for i in range(len(X)):
    review = re.sub('[^a-zA-Z]', ' ', X.iloc[i])
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [16]:
onehot_repr = [one_hot(words, voc_size) for words in corpus]


In [71]:
onehot_repr

[[506,
  3541,
  2493,
  3887,
  3226,
  2704,
  3087,
  3286,
  2167,
  1072,
  2971,
  4569,
  2119,
  2426,
  2704,
  824,
  4874,
  2426],
 [506,
  257,
  2493,
  2992,
  2114,
  4593,
  4297,
  2007,
  2992,
  3087,
  3286,
  798,
  1084,
  1910,
  4540,
  3724,
  3959,
  2971,
  4569,
  2119,
  2960,
  1027,
  2426],
 [2823, 3121, 2339],
 [3541, 2440, 4659],
 [2056, 2915],
 [1504, 4255, 214],
 [3337, 2750],
 [4794, 4607, 2390, 2339],
 [3324, 1970, 92, 2339],
 [2988, 2855, 2056, 3689],
 [3191, 3061],
 [506,
  4524,
  3887,
  1835,
  4005,
  2114,
  3065,
  4593,
  1653,
  2119,
  2960,
  1027,
  2426,
  1546,
  3480],
 [2426,
  3887,
  1037,
  4433,
  972,
  917,
  4978,
  1731,
  2653,
  4002,
  409,
  1546,
  470,
  576,
  4545,
  2493],
 [506, 3541, 2493, 4569, 506, 3541, 2426],
 [506,
  3541,
  2493,
  2704,
  506,
  3887,
  3087,
  3286,
  4005,
  506,
  4593,
  3887,
  4545,
  824,
  2971,
  4569,
  2119,
  2426,
  2704,
  824,
  4874,
  2426],
 [506,
  3541,
  2493,
  2704,

In [17]:
sent_length = 20
embedded_docs = pad_sequences(onehot_repr, padding='pre', maxlen=sent_length)


In [18]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)


In [19]:
y = to_categorical(y, num_classes=len(np.unique(y)))

In [20]:
# Convert to numpy arrays
X_final = np.array(embedded_docs)
y_final = np.array(y)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)


In [87]:
embedding_vector_features = 40
model = Sequential()
model.add(Embedding(voc_size, embedding_vector_features, input_length=sent_length))
model.add(LSTM(100))
model.add(Dense(46, activation='softmax'))  # Number of classes as output
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [88]:
print(model.summary())


Model: "sequential_13"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_13 (Embedding)    (None, 20, 40)            200000    
                                                                 
 lstm_13 (LSTM)              (None, 100)               56400     
                                                                 
 dense_13 (Dense)            (None, 46)                4646      
                                                                 
Total params: 261046 (1019.71 KB)
Trainable params: 261046 (1019.71 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [89]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


X_train shape: (45597, 20)
y_train shape: (45597,)
X_test shape: (22459, 20)
y_test shape: (22459,)


In [90]:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=64)


Epoch 1/10
713/713 [==============================] - 14s 17ms/step - loss: 2.5856 - accuracy: 0.3196 - val_loss: 1.6730 - val_accuracy: 0.5479
Epoch 2/10
713/713 [==============================] - 11s 15ms/step - loss: 1.3323 - accuracy: 0.6419 - val_loss: 1.2662 - val_accuracy: 0.6547
Epoch 3/10
713/713 [==============================] - 11s 15ms/step - loss: 0.9901 - accuracy: 0.7283 - val_loss: 1.1294 - val_accuracy: 0.6918
Epoch 4/10
713/713 [==============================] - 12s 17ms/step - loss: 0.8171 - accuracy: 0.7726 - val_loss: 1.0799 - val_accuracy: 0.7049
Epoch 5/10
713/713 [==============================] - 11s 16ms/step - loss: 0.7019 - accuracy: 0.8016 - val_loss: 1.0366 - val_accuracy: 0.7193
Epoch 6/10
713/713 [==============================] - 11s 15ms/step - loss: 0.6172 - accuracy: 0.8233 - val_loss: 1.0376 - val_accuracy: 0.7216
Epoch 7/10
713/713 [==============================] - 11s 15ms/step - loss: 0.5467 - accuracy: 0.8423 - val_loss: 1.0203 - val_accuracy:

In [94]:
y_pred_prob = model.predict(X_test)

702/702 [==============================] - 3s 4ms/step


In [95]:
y_pred1 = np.argmax(y_pred_prob, axis=1)

In [96]:
print("Predicted class labels:", y_pred1[:10])

Predicted class labels: [ 8 37 39  9 17 24 10 36 39  8]


In [97]:
from sklearn.metrics import confusion_matrix

In [98]:
confusion_matrix(y_test,y_pred1)

array([[   5,    0,    0, ...,    0,    0,    0],
       [   0,  502,    7, ...,    8,    2,   16],
       [   0,    9,  428, ...,    6,    7,   50],
       ...,
       [   0,    4,    2, ...,  640,   10,   31],
       [   0,    1,    6, ...,   20,  283,   10],
       [   0,   27,   30, ...,   35,    4, 1623]], dtype=int64)

In [99]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred1)

0.7344494412039717

In [100]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred1))

              precision    recall  f1-score   support

           0       0.56      0.71      0.63         7
           1       0.59      0.68      0.63       743
           2       0.68      0.65      0.66       659
           3       0.77      0.67      0.71       729
           4       0.73      0.72      0.73       538
           5       0.74      0.69      0.72       173
           6       0.77      0.77      0.77       283
           7       0.65      0.57      0.61       150
           8       0.77      0.83      0.80      1041
           9       0.67      0.71      0.69       318
          10       0.63      0.74      0.68       302
          11       0.67      0.59      0.63       448
          12       0.65      0.80      0.72       257
          13       0.63      0.54      0.58       604
          14       0.70      0.59      0.64       785
          15       0.62      0.62      0.62       225
          16       0.88      0.99      0.93        74
          17       0.65    

In [120]:
def preprocess_description(description, ps, voc_size, sent_length):
    review = re.sub('[^a-zA-Z]', ' ', description)
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    onehot_repr = one_hot(review, voc_size)
    embedded_docs = pad_sequences([onehot_repr], padding='pre', maxlen=sent_length)
    return np.array(embedded_docs)

In [121]:
def predict_class(description, model, ps, label_encoder, voc_size, sent_length):

    processed_description = preprocess_description(description, ps, voc_size, sent_length)
    y_pred_prob = model.predict(processed_description)
    y_pred = np.argmax(y_pred_prob, axis=1)
    predicted_class = label_encoder.inverse_transform(y_pred)[0]
    
    return predicted_class

In [135]:
sample_description = "Smart thermostats with Wi-Fi connectivity"
predicted_class = predict_class(sample_description, model, ps, label_encoder, voc_size, sent_length)
print(f"The predicted class for the given description is: {predicted_class}")

1/1 [==============================] - 0s 18ms/step
The predicted class for the given description is: 28
